# 03 · Task master compilation (Section 3.1.3, Table B1)

Consolidates every task-level variable used downstream into **one row per register task**, built on the final R2R task register from phase 1 (Appendix A, Table A2). Everything is matched by `task_id`:

| Column (Table B1) | Source |
|---|---|
| `task_id`, `task_statement`, `subprocess` | final task register (`final_label` → `subprocess`) |
| `occupation_id`, `occupation_title` | O\*NET task statements (each task belongs to exactly one occupation) |
| `wage_hourly_mean`, `wage_annual_mean` | BLS OEWS May 2025, national cross-industry, detailed occupation, matched on 6-digit SOC (task value) |
| `ridge_task_years` | notebook 02, `predicted_task_years` |
| `task_time_fraction` | notebook 01, `cross_model_mean_hours` (weekly hours, 3-model mean; renormalised within scope only at the point of use) |
| `frequency_occ_per_year` | O\*NET task ratings, Frequency of Task (FT) categories → expected occurrences per year (not used in the final constructs, Section 3.2.3) |
| `beta` | Eloundou et al. (2024) task-level exposure β, merged last |

Run **after 01 and 02**.

| Output (`outputs/03_task_master/`) | Contents |
|---|---|
| `task_master.csv` | the task master (212 rows) → **handed off** to `final_pipeline/data/` |
| `task_master_checks.csv` | non-missing count, min, mean and max of every column, all tasks and in-scope only |
| `subprocess_summary.csv` | in-scope tasks per subprocess with mean wage, ridge years, beta and total weekly hours |

The register is also handed off, because `final_pipeline/05` uses it to split Table D6 into in-scope tasks.

In [1]:
import numpy as np
import pandas as pd

import prep_common as pc

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## Configuration

**Frequency scale.** O\*NET reports, for each task, the percentage of incumbents choosing each of seven frequency categories (`task_categories.xlsx`). Each category is converted to occurrences per year, and the task's value is the share-weighted mean (shares rescaled to sum to 1, because O\*NET's rounded percentages sum to 99.98–100.02).

In [2]:
FT_OCCURRENCES_PER_YEAR = {
    1: 1,      # Yearly or less
    2: 3.5,    # More than yearly
    3: 25,     # More than monthly
    4: 114,    # More than weekly
    5: 250,    # Daily
    6: 750,    # Several times daily
    7: 2000,   # Hourly or more
}

# OEWS national file: one row per occupation at the "detailed" level, all industries
OEWS_FILTER = {"AREA": 99, "NAICS": "000000", "O_GROUP": "detailed"}

TASK_MASTER_COLS = ["task_id", "task_statement", "subprocess", "occupation_id", "occupation_title",
                    "wage_hourly_mean", "wage_annual_mean", "ridge_task_years", "task_time_fraction",
                    "frequency_occ_per_year", "beta"]

## 1. Register and occupations

In [3]:
register = pc.load_register()
tm = register[["task_id", "task_statement", "final_label"]].rename(columns={"final_label": "subprocess"})

onet = pc.load_onet_tasks()
occ = (onet[[pc.ONET_TASKID_COL, pc.ONET_SOC_COL, pc.ONET_TITLE_COL]]
       .rename(columns={pc.ONET_TASKID_COL: "task_id", pc.ONET_SOC_COL: "occupation_id",
                        pc.ONET_TITLE_COL: "occupation_title"})
       .drop_duplicates())
n_occ = occ[occ["task_id"].isin(tm["task_id"])].groupby("task_id").size()
if (n_occ > 1).any():
    raise ValueError(f"{(n_occ > 1).sum()} register tasks belong to more than one O*NET occupation")
tm = tm.merge(occ, on="task_id", how="left", validate="one_to_one")
print(f"{len(tm)} tasks, {tm['occupation_id'].isna().sum()} without an O*NET occupation")
print(tm.groupby(["occupation_id", "occupation_title"]).size().rename("tasks").to_string())

212 tasks, 0 without an O*NET occupation
occupation_id  occupation_title                            
11-1011.00     Chief Executives                                31
11-3031.00     Financial Managers                              17
11-3031.01     Treasurers and Controllers                      22
13-1051.00     Cost Estimators                                 14
13-2011.00     Accountants and Auditors                        29
13-2031.00     Budget Analysts                                 13
13-2082.00     Tax Preparers                                   12
15-1242.00     Database Administrators                         18
43-3021.00     Billing and Posting Clerks                      28
43-3031.00     Bookkeeping, Accounting, and Auditing Clerks    28


## 2. Task value: OEWS mean wages

In [4]:
oews = pd.read_excel(pc.OEWS_WAGE_FILE, dtype={"NAICS": str, "OCC_CODE": str})
oews = oews[(oews["AREA"] == OEWS_FILTER["AREA"]) & (oews["NAICS"] == OEWS_FILTER["NAICS"])
            & (oews["O_GROUP"] == OEWS_FILTER["O_GROUP"])]
wages = (oews[["OCC_CODE", "H_MEAN", "A_MEAN"]]
         .assign(wage_hourly_mean=lambda d: pd.to_numeric(d["H_MEAN"], errors="coerce"),   # '*' / '#' -> NaN
                 wage_annual_mean=lambda d: pd.to_numeric(d["A_MEAN"], errors="coerce"))
         .set_index("OCC_CODE")[["wage_hourly_mean", "wage_annual_mean"]])
if wages.index.duplicated().any():
    raise ValueError("OEWS filter returned more than one row for an occupation")
soc = pc.soc6(tm["occupation_id"])
tm["wage_hourly_mean"] = soc.map(wages["wage_hourly_mean"])
tm["wage_annual_mean"] = soc.map(wages["wage_annual_mean"])
print(tm.groupby(["occupation_id", "occupation_title"])[["wage_hourly_mean", "wage_annual_mean"]].first().to_string())

                                                            wage_hourly_mean  wage_annual_mean
occupation_id occupation_title                                                                
11-1011.00    Chief Executives                                        129.63          269630.0
11-3031.00    Financial Managers                                       89.86          186910.0
11-3031.01    Treasurers and Controllers                               89.86          186910.0
13-1051.00    Cost Estimators                                          41.06           85390.0
13-2011.00    Accountants and Auditors                                 45.56           94750.0
13-2031.00    Budget Analysts                                          46.33           96370.0
13-2082.00    Tax Preparers                                            29.30           60930.0
15-1242.00    Database Administrators                                  52.93          110090.0
43-3021.00    Billing and Posting Clerks          

## 3. Ridge expertise (02) and time fraction (01)

In [5]:
ridge = pc.read_output("ridge", "ridge_task_complexity.csv", dtype={pc.ONET_TASKID_COL: str})
ridge[pc.ONET_TASKID_COL] = pc.normalize_id(ridge[pc.ONET_TASKID_COL])
tm["ridge_task_years"] = tm["task_id"].map(ridge.set_index(pc.ONET_TASKID_COL)["predicted_task_years"])

tf = pc.read_output("time_fraction", "elicited_time_fractions.csv")
tm["task_time_fraction"] = tm["task_id"].map(tf.set_index("task_id")["cross_model_mean_hours"])
print("missing ridge_task_years:", tm["ridge_task_years"].isna().sum(),
      "| missing task_time_fraction:", tm["task_time_fraction"].isna().sum())

missing ridge_task_years: 0 | missing task_time_fraction: 0


## 4. Task frequency (O\*NET FT scale)

In [6]:
cats = pd.read_excel(pc.ONET_CATEGORIES_FILE)
print(cats.assign(occurrences_per_year=cats["Category"].map(FT_OCCURRENCES_PER_YEAR))
      [["Category", "Category Description", "occurrences_per_year"]].to_string(index=False))

ratings = pd.read_excel(pc.ONET_RATINGS_FILE)
ft = ratings[ratings["Scale ID"] == "FT"].copy()
ft["task_id"] = pc.normalize_id(ft[pc.ONET_TASKID_COL])
ft = ft[ft["task_id"].isin(tm["task_id"])]
shares = ft.pivot_table(index=["task_id", pc.ONET_SOC_COL], columns="Category", values="Data Value")
shares.columns = shares.columns.astype(int)
shares = shares.div(shares.sum(axis=1), axis=0)
freq = (shares * pd.Series(FT_OCCURRENCES_PER_YEAR)).sum(axis=1).rename("frequency_occ_per_year").reset_index()
freq = freq.rename(columns={pc.ONET_SOC_COL: "occupation_id"})
tm = tm.merge(freq, on=["task_id", "occupation_id"], how="left", validate="one_to_one")
no_ft = tm[tm["frequency_occ_per_year"].isna()]
print(f"\n{len(tm) - len(no_ft)} tasks with FT ratings; {len(no_ft)} without "
      f"({(no_ft['subprocess'] != pc.OUT_OF_SCOPE).sum()} of them in scope)")

 Category Category Description  occurrences_per_year
        1       Yearly or less                   1.0
        2     More than yearly                   3.5
        3    More than monthly                  25.0
        4     More than weekly                 114.0
        5                Daily                 250.0
        6  Several times daily                 750.0
        7       Hourly or more                2000.0



203 tasks with FT ratings; 9 without (5 of them in scope)


## 5. Eloundou et al. (2024) β, merged last

In [7]:
elo = pd.read_csv(pc.ELOUNDOU_FILE, sep="\t")
elo["task_id"] = pc.normalize_id(elo[pc.ONET_TASKID_COL])
if elo["task_id"].duplicated().any():
    raise ValueError("Task ID is not unique in the Eloundou file")
tm["beta"] = tm["task_id"].map(elo.set_index("task_id")["beta"])
print("beta distribution (in scope):")
print(tm.loc[tm["subprocess"] != pc.OUT_OF_SCOPE, "beta"].value_counts(dropna=False).sort_index().to_string())

beta distribution (in scope):
beta
0.0     1
0.5    60
1.0    30


## 6. Checks and save

In [8]:
tm = tm[TASK_MASTER_COLS]
assert not tm["task_id"].duplicated().any(), "task_id must be unique"
assert len(tm) == len(register), "one row per register task"
in_scope = tm["subprocess"] != pc.OUT_OF_SCOPE
print(f"{len(tm)} tasks: {in_scope.sum()} in scope, {(~in_scope).sum()} OUT_OF_SCOPE")

num_cols = ["wage_hourly_mean", "wage_annual_mean", "ridge_task_years", "task_time_fraction",
            "frequency_occ_per_year", "beta"]
checks = []
for scope, d in [("all", tm), ("in_scope", tm[in_scope])]:
    for c in TASK_MASTER_COLS:
        row = {"scope": scope, "column": c, "n_rows": len(d), "non_missing": int(d[c].notna().sum())}
        if c in num_cols:
            row.update(min=d[c].min(), mean=d[c].mean(), max=d[c].max())
        checks.append(row)
checks = pd.DataFrame(checks)
pc.save_csv(checks, "task_master", "task_master_checks.csv")

sp = (tm[in_scope].groupby("subprocess")
      .agg(n_tasks=("task_id", "size"), n_occupations=("occupation_id", "nunique"),
           mean_wage_hourly=("wage_hourly_mean", "mean"), mean_ridge_years=("ridge_task_years", "mean"),
           mean_beta=("beta", "mean"), total_weekly_hours=("task_time_fraction", "sum"))
      .reset_index())
pc.save_csv(sp, "task_master", "subprocess_summary.csv")
pc.save_csv(tm, "task_master", "task_master.csv")
checks[checks["scope"] == "in_scope"].round(2)

212 tasks: 91 in scope, 121 OUT_OF_SCOPE
[write] outputs/03_task_master/task_master_checks.csv  (22 rows)
[write] outputs/03_task_master/subprocess_summary.csv  (6 rows)
[write] outputs/03_task_master/task_master.csv  (212 rows)


,scope,column,n_rows,non_missing,min,mean,max
11,in_scope,task_id,91,91,NaN,NaN,NaN
12,in_scope,task_statement,91,91,NaN,NaN,NaN
13,in_scope,subprocess,91,91,NaN,NaN,NaN
14,in_scope,occupation_id,91,91,NaN,NaN,NaN
15,in_scope,occupation_title,91,91,NaN,NaN,NaN
16,in_scope,wage_hourly_mean,91,91,24.55,49.44,129.63
17,in_scope,wage_annual_mean,91,91,51070.00,102834.29,269630.00
18,in_scope,ridge_task_years,91,91,9.30,14.24,17.11
19,in_scope,task_time_fraction,91,91,0.58,3.50,11.00
20,in_scope,frequency_occ_per_year,91,86,6.27,268.38,1162.94


## Hand-off to the analysis phase

In [9]:
pc.handoff("task_master", "task_master.csv")
dst = pc.ANALYSIS_DATA_DIR / pc.REGISTER_FILE.name
dst.write_bytes(pc.REGISTER_FILE.read_bytes())
print(f"[handoff] {pc.show_path(pc.REGISTER_FILE)} -> {pc.show_path(dst)}")

[handoff] outputs/03_task_master/task_master.csv -> ../final_pipeline/data/task_master.csv
[handoff] inputs/final_task_register.csv -> ../final_pipeline/data/final_task_register.csv
